In [1]:
#!/usr/bin/env python3
import rospy
from actionlib import SimpleActionClient
from assignment_2_2024.msg import PlanningAction, PlanningGoal, PlanningFeedback
from nav_msgs.msg import Odometry
from geometry_msgs.msg import Pose, Twist
from std_msgs.msg import Float32
from sensor_msgs.msg import LaserScan
import ipywidgets as widgets
from ipywidgets import Button, Layout, HBox, VBox
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
import random, genpy
import yaml
from gazebo_msgs.srv import SetModelState

from gazebo_msgs.msg import ModelState
rospy.init_node("target_client")


In [2]:
# Variabili per la gestione dei target e dei grafici
target_given_list = []
target_reached_list = []
random_spawn_position = []
target_cancelled_list = []
target_random_list = None
target_times = {}  # used to store times of targets
times_recap = widgets.Output()
num_targ = 88

# processing data
pose = Pose()
goal = PlanningGoal()
stat = ""
vel = Twist()
xdata, ydata = [], []
curr_x = 0.0
curr_y = 0.0
clos_obstac = 0.0
running = False
start_time = None


lab_feed = widgets.Label(value="")

# Box to see the list of targets
given_list_box = widgets.VBox([])
reached_list_box = widgets.VBox([])
cancelled_list_box = widgets.VBox([])

# widgets buttons and texts
start_b = None
cancel_b = None
rand_start_b = None
x_input = None
y_input = None
text_x = None
text_y = None
text_ob = None
run_out_time_flag = False

# floag to avoid to miss some targets
target_completed = False
done_targets = False

# debug
feed = widgets.Label(value="feed:")
sono_qui = widgets.Label(value="")

# Definizione delle callback per ROS
client = SimpleActionClient("reaching_goal", PlanningAction)
client.wait_for_server()


# texts for data of the closest obstacle and position
text_x = widgets.Text(value="0.0", disabled=True, layout=widgets.Layout(width='80px'))
text_y = widgets.Text(value="0.0", disabled=True, layout=widgets.Layout(width='80px'))
text_ob = widgets.Text(value="0.0", disabled=True, layout=widgets.Layout(width='80px'))


# texts of the closest obstacle and position
label_pos = widgets.HTML(value='<h2 style="margin:0px; padding-right:10px;">Position (x,y):</h2>', layout=widgets.Layout(display='flex', align_items='center'))
label_ob = widgets.HTML(value='<h2 style="margin:0px; padding-right:10px;">Closest obstacle:</h2>', layout=widgets.Layout(display='flex', align_items='center'))
label_metres = widgets.HTML(value='<h2 style="margin:0px; padding-left:10px;">metres</h2>', layout=widgets.Layout(display='flex', align_items='center'))


In [3]:
def pos_vel_callback(msg):
    global vel, xdata, ydata, curr_x, curr_y, text_x, text_y
    # obtain the position and publishes it
    vel = msg.twist.twist
    xdata.append(msg.pose.pose.position.x)
    ydata.append(msg.pose.pose.position.y)
    curr_x = msg.pose.pose.position.x
    curr_y = msg.pose.pose.position.y
    text_x.value = f'{curr_x:.3f}'
    text_y.value = f'{curr_y:.3f}'

def feedback_callback(feedback_: PlanningFeedback) -> None:
    global stat
    stat = feedback_.feedback.stat
    process_feedback()

def laser_callback(msg):
    global clos_obstac, text_ob, publish_distance # Assuming publish_distance is defined elsewhere
    # obtain the obstacles and compute the min distance
    min_distance = min(msg.ranges)
    closest_obstacle_msg = Float32() # Assuming Float32 is imported from std_msgs.msg
    closest_obstacle_msg.data = min_distance
    publish_distance.publish(closest_obstacle_msg)
    clos_obstac = min_distance
    text_ob.value = f'{clos_obstac:.3f}'


In [4]:
reached_w = widgets.Label(value="")
cancelled_w = widgets.Label(value="")
run_out_time_flag = False  

def process_feedback():
    global stat, num_targ, target_completed, target_random_list, target_reached_list,target_given_list
    global given_list_box, reached_list_box, start_time, target_cancelled_list
    global cancelled_list_box, cancel_b, start_b, rand_start_b, done_targets
    global sono_qui, target_times, run_out_time_flag

    feed.value = stat
    target_id = str(num_targ)
    max_time = 5*60
    if stat == "Target reached!" :#and not target_completed:
        target_completed = True
        sono_qui.value = " process_feedback: reached"

        current_target = target_given_list[-1]
        target_reached_list.append(current_target)
        update_target_list("reached")

        end_time = rospy.Time.now()
        if target_id in target_times:
            duration = end_time - target_times[target_id]['start']
            sono_qui.value = f' process_feedback: durata = {duration.to_sec()} sec'
            run_out_time_flag = False  
            load_result_target(target_id, end_time, duration, "True")

        # Passa al prossimo target se disponibile
        if target_random_list:
            random_pos = teleport()
            next_target = target_random_list.pop(0)
            sono_qui.value = " process_feedback: nuovo target"
            do_goal(next_target[0], next_target[1],random_pos)
            run_out_time_flag = False 
            target_completed = False
        else:
            # Fine dei target
            sono_qui.value = " process_feedback: tutti i target completati"
            del_goal()
            target_random_list = []
            cancel_b.disabled = True
            start_b.disabled = False
            rand_start_b.disabled = False
            update_times_recap()
            a = teleport()
            run_out_time_flag = False 
            done_targets = True


    elif start_time is not None:
        now = rospy.Time.now()
        elapsed = now - start_time
        feed.value = f'value : {elapsed.to_sec()}'

        reason_to_cancel = None
        if stat == "Target cancelled!":
            reason_to_cancel = "explicit"
        elif (elapsed.to_sec() >= max_time) and (stat != "Target reached!" and stat != "Target cancelled!"):
            reason_to_cancel = "timeout"

        # Evita doppie esecuzioni
        if reason_to_cancel:
            
            if reason_to_cancel == "explicit" and run_out_time_flag == False:
                del_goal()
                
                current_target = target_given_list[-1]
                target_cancelled_list.append(current_target)
                update_target_list("cancelled")
                start_time = None
                sono_qui.value = " process_feedback: cancelled explicitly"
                cancel_b.disabled = True
                start_b.disabled = False
                rand_start_b.disabled = False
                target_random_list = []
                done_targets = True
                num_targ = 0

            elif reason_to_cancel == "timeout" :
                del_goal()
                
                current_target = target_given_list[-1]
                target_cancelled_list.append(current_target)
                update_target_list("cancelled")
                start_time = None
                
                sono_qui.value = f" process_feedback: timeout, dim : {len(target_random_list)}"
                over_time = "False"
                max_time_ = genpy.Duration(max_time)
                load_result_target(target_id, rospy.Time.now(), max_time_, over_time)
                random_pos_teleported = teleport()
                run_out_time_flag = True 
                sono_qui.value =( f" process_feedback:  {target_random_list} \n"
                                f" process_feedback:  {target_given_list} \n"
                                 f" process_feedback:  {target_cancelled_list} \n"
                                )
                if target_random_list:  
                    next_target = target_random_list.pop(0) 
                    sono_qui.value = f" process_feedback: invio goal {next_target}"
                    do_goal(next_target[0], next_target[1],  random_pos_teleported)
                    sono_qui.value = " process_feedback: mandato"
                    target_completed = False
                    elapsed = genpy.Duration(0)

                    


In [5]:
laod_ers_w = widgets.Label(value = "")

def load_result_target(target_id, end_time, duration, run_out_time):
    global target_times
    laod_ers_w.value=f"load_result_target: enter, id: {target_id},{end_time},{duration},{run_out_time}"
    try:
        if target_id in target_times:

            target_times[target_id]['end'] = end_time
            target_times[target_id]['duration'] = duration
            target_times[target_id]['validity'] = run_out_time
            laod_ers_w.value= f"load_result_target: enter if, {target_times[target_id]}"

            # Prepare data to save (converted to seconds)
            times = target_times[target_id]
            laod_ers_w.value= f":tmes, \n{times}"


            data_to_save = {
                'id': times['id'],
                'spawn_position': times['spawn_position'] if times['spawn_position'] else None,
                'target': times['target'],
                'start': times['start'].to_sec() if times['start'] else None,
                'end': times['end'].to_sec() if times['end'] else None,
                'duration': times['duration'].to_sec() if times['duration'] else None,
                'validity': run_out_time
            }
            laod_ers_w.value = f"load_result_target:data save {data_to_save}"

            with open("results_targets1.yaml", "a") as f:
                yaml.dump({f"target_{target_id}": data_to_save}, f, explicit_start=True)
            laod_ers_w.value = "load_result_target: loaded"
        

        else:
            laod_ers_w.value= f"load_result_target: Error: Target ID {target_id} not found in target_times."
    except Exception as e:
        laod_ers_w.value = (
            f"load_result_target: ERROR during save: {str(e)}\n"
            f"target_id: {target_id} ({type(target_id).__name__})\n"
            f"end_time: {end_time} ({type(end_time).__name__})\n"
            f"duration: {duration} ({type(duration).__name__})\n"
            f"run_out_time: {run_out_time} ({type(run_out_time).__name__})"
        )




In [6]:
# teleport_widget = widgets.Label(value= "")
# def teleport(): 
#     global set_state,random_spawn_position
#     teleport_widget.value = "teleport: inside teleport"
#     try:
#         teleport_widget.value = "teleport: inside teleport2"
#         if random_spawn_position:
#             wandom_pos_spawn = random_spawn_position.pop(0)
#             pos_x, pos_y = wandom_pos_spawn
#             teleport_widget.value = f'teleport: positions {pos_x}, {pos_y}'
#             state_msg = ModelState()
#             state_msg.model_name = 'robot1'
#             state_msg.pose.position.x =  pos_x
#             state_msg.pose.position.y = pos_y
#             state_msg.pose.position.z = 0.0
#             state_msg.pose.orientation.x = 0.0
#             state_msg.pose.orientation.y = 0.0
#             state_msg.pose.orientation.z = 0.0
#             state_msg.pose.orientation.w = 0.0

#             state_msg.twist.linear.x = 0.0
#             state_msg.twist.linear.y = 0.0
#             state_msg.twist.linear.z = 0.0
#             state_msg.twist.angular.x = 0.0
#             state_msg.twist.angular.y = 0.0
#             state_msg.twist.angular.z = 0.0

#             state_msg.reference_frame = 'world'

#             # Chiama il servizio usando rospy
#             req = SetModelStateRequest()
#             req.model_state = state_msg
#             try:
#                 resp = set_state(req)  # Chiama il servizio e ottieni la risposta
#                 teleport_widget.value = f"teleport: Success: {resp.success}, Status: {resp.status_message}"
#             except rospy.ServiceException as e:
#                 teleport_widget.value = f"teleport: Service call failed: {e}"
#         else:
#             teleport_widget.value = "No more random spawn positions available."
#     except rospy.ServiceException as e:
#         teleport_widget.value = f"teleport: ERROR during teleport: {e}"
teleport_widget = widgets.Label(value= "")
def teleport():
    global random_spawn_position, set_state # Assuming set_state is a ROS service proxy
    teleport_widget.value = "teleport: inside teleport"
    try:
        teleport_widget.value = "teleport: inside teleport2"
        if random_spawn_position:
            random_pos_spawn = random_spawn_position.pop(0) # Typo in original code: 'run_out_of_time' should likely be 'random_spawn_position'
            pos_x, pos_y = random_pos_spawn
            teleport_widget.value = f'teleport: positions {pos_x}, {pos_y}'
            state_msg = ModelState() # Assuming ModelState is imported from gazebo_msgs.msg
            state_msg.model_name = 'robot1'
            state_msg.pose.position.x = pos_x
            state_msg.pose.position.y = pos_y
            state_msg.pose.position.z = 0.0
            state_msg.pose.orientation.x = 0.0
            state_msg.pose.orientation.y = 0.0
            state_msg.pose.orientation.z = 0.0
            state_msg.pose.orientation.w = 0.0

            state_msg.twist.linear.x = 0.0
            state_msg.twist.linear.y = 0.0
            state_msg.twist.linear.z = 0.0
            state_msg.twist.angular.x = 0.0
            state_msg.twist.angular.y = 0.0
            state_msg.twist.angular.z = 0.0

            state_msg.reference_frame = 'world'

            resp = set_state(state_msg)
            teleport_widget.value = f"teleport: Success: {resp.success}, Status: {resp.status_message}"
            
            return random_pos_spawn
        else:
            teleport_widget.value = f"No more random spawn positions available."
    except Exception as e:
        teleport_widget.value = f"teleport: ERROR during teleport : {str(e)}"

In [7]:
def update_times_recap():
    global target_times, times_recap
    # gives me a recap , can be seen in yaml file
    with times_recap:
        times_recap.clear_output()
        for target_id, times in target_times.items():
            start = times['start'].to_sec() if times['start'] else None
            end = times['end'].to_sec() if times['end'] else None
            duration = times['duration'].to_sec() if times['duration'] else None
            run_out_of_time = times['run_out_of_time']
            print(f"Target ID: {target_id}")
            print(f"Target: {times['target']}")
            print(f"Inizio:    {start}")
            print(f"Fine:      {end}")
            print(f"Durata:    {duration} s")
            print(f"Run out of time: {run_out_of_time}\n")


In [8]:
def update_plots(frame):
    global target_reached_list, target_cancelled_list, target_given_list, ax1, ln, xdata, ydata # Assuming ax1 and ln are defined elsewhere for plotting
    # updating pie chart and the map
    if len(target_reached_list) != 0 or len(target_cancelled_list) != 0:
        ax1.clear()
        sizes = [len(target_reached_list), len(target_cancelled_list)]
        labels = ['Reached', 'Cancelled']
        ax1.pie(sizes, labels=labels, autopct='%1.1f%%')
        ax1.set_title(f'Reached vs Cancelled \n sent: {len(target_given_list)}, reached: {len(target_reached_list)}, cancelled: {len(target_cancelled_list)}', fontsize=10)

    ln.set_data(xdata, ydata)
    return ln,



def update_target_list(status):
    global target_reached_list, reached_list_box, target_cancelled_list, cancelled_list_box, target_given_list, given_list_box, sono_qui
    # sono_qui.value = f'5. stat : {status}'

    # if reaches I change the color of the target in green
    if status == "reached":
        if target_reached_list:
            new_target_rea = widgets.HTML(f'<span style="color: black;">{target_reached_list[-1]}</span>')
            reached_list_box.children += (new_target_rea,)
            new_target_can = widgets.HTML(f'<br>')
            cancelled_list_box.children += (new_target_can,)
            if target_given_list:
                given_list_box.children[-1].value = f'<span style="color: green;">{target_given_list[-1]}</span>'
        else:
            print("Error: target_reached_list is empty when trying to update the reached list.")

    # if cancel I change the color of the target in red
    elif status == "cancelled":
        if target_cancelled_list:
            new_target_rea = widgets.HTML(f'<br>')
            reached_list_box.children += (new_target_rea,)
            new_target_can = widgets.HTML(f'<span style="color: black;">{target_cancelled_list[-1]}</span>')
            cancelled_list_box.children += (new_target_can,)
            if target_given_list:
                given_list_box.children[-1].value = f'<span style="color: red;">{target_given_list[-1]}</span>'
        else:
            print("Error: target_cancelled_list is empty when trying to update the cancelled list.")

In [9]:
def do_goal(x, y, random_pos_teleported = None):
    global num_targ, target_given_list, given_list_box, client, goal, start_time, target_times, done_targets
    done_targets = False
    # building the target with the element given
    pose = Pose() # Create a local Pose object
    pose.position.x = float(x)
    pose.position.y = float(y)
    target = {'aim_x': pose.position.x, 'aim_y': pose.position.y}

    # adding the target to the target list
    target_given_list.append(target)

    # updating the widget showing th current target
    new_target = widgets.HTML(f'<span style="color: black;">{target}</span>')
    given_list_box.children += (new_target,)

    # building and sending the goal
    goal_msg = PlanningGoal() # Create a local PlanningGoal object
    goal_msg.target_pose.pose = pose
    client.send_goal(goal_msg)

    # storing the initial starting moment
    start_time = rospy.Time.now()
    num_targ += 1
    
    
    
    if random_pos_teleported is not None:
        x_pos , y_pos = random_pos_teleported
    else:
        x_pos =0
        y_pos =0
        
    
    pose_spawn = Pose()
    pose_spawn.position.x = float(x_pos)
    pose_spawn.position.y = float(y_pos)
    spwned_pos = {'x':  pose_spawn.position.x, 'y':  pose_spawn.position.y}
    
    
    # storing the data in the dictionary to associate to each target start, end and duration time
    target_times[str(num_targ)] = {'id': num_targ,
                                   'spawn_position' :spwned_pos,
                                   'target': target,
                                   'start': start_time,
                                   'end': 0,
                                   'duration': 0,
                                   'validity': "False"
                                   }

def del_goal():
    global client
    client.cancel_all_goals()


In [10]:
def on_start_button_clicked(b):
    global start_b, cancel_b, x_input, y_input
    start_b.disabled = True
    cancel_b.disabled = False
    do_goal(x_input.value, y_input.value)


def on_start_rand_button_clicked(b):
    global start_b, cancel_b, target_random_list, done_targets
    # print("qui")
    # print(f'lungh vettore  {len(target_random_list)}, cond :{done_targets}')
    # if cancelled or the targets have already been reached, then i cannot play "random start" unless I give other targets
    if done_targets == True and len(target_random_list) == 0:
        return
    start_b.disabled = True
    cancel_b.disabled = False
    if target_random_list:
        targ = target_random_list.pop(0)
        do_goal(targ[0], targ[1])
    else:
        print("Error: target_random_list is empty.")

def on_cancel_button_clicked(b):
    global cancel_b, start_b, x_input, y_input
    run_out_time_flag = False
    cancel_b.disabled = True
    start_b.disabled = False
    x_input.disabled = False
    y_input.disabled = False
    x_input.value = 0.0
    y_input.value = 0.0
    del_goal()

def update_widgets(event):
    global curr_x, curr_y, clos_obstac, text_x, text_y, text_ob
    text_x.value = f'{curr_x:.4f}'
    text_y.value = f'{curr_y:.4f}'
    text_ob.value = f'{clos_obstac:.4f}'
# Input per il target



In [11]:

# ubscrbers and publishers
rospy.Subscriber("/odom", Odometry, pos_vel_callback)
rospy.Subscriber("/reaching_goal/feedback", PlanningFeedback, feedback_callback)
rospy.Subscriber('/scan', LaserScan, laser_callback)
publish_distance = rospy.Publisher('closest_obstacle', Float32, queue_size=10)
rospy.wait_for_service('/gazebo/set_model_state')
set_state = rospy.ServiceProxy('/gazebo/set_model_state', SetModelState)


# charts and graphic
%matplotlib widget
plt.ioff()
fig, (ax2, ax1) = plt.subplots(1, 2, figsize=(9, 5))
fig.patch.set_visible(False)
ax1.set_title(
    f'Reached vs Cancelled\nsent: {len(target_given_list)}, '
    f'reached: {len(target_reached_list)}, cancelled: {len(target_cancelled_list)}',
    fontsize=10
)
ax2.set_title("Live trajectory")
ax2.grid()
ln, = ax2.plot([], [], 'r-')
ax2.set_xlim(10, -10)
ax2.set_ylim(10, -10)
plt.ion()
ani = FuncAnimation(fig, update_plots, interval=100)


message_label = widgets.HTML(
    value='<h2 style="margin:0px; padding-left:0px;">Set target (x,y):</h2>',
    layout=widgets.Layout(display='flex', align_items='center')
)
x_input = widgets.FloatText(value=0.0, layout=widgets.Layout(width='80px'))
y_input = widgets.FloatText(value=0.0, layout=widgets.Layout(width='80px'))
box_input = HBox([message_label, x_input, y_input], layout=widgets.Layout(justify_content='flex-start', align_items='center', width='100%'))

del_goal()

# # declaring number of experiments
# num_departures = 3
random_positions = None #[(random.randint(-9, 9), random.randint(2, 3)) for _ in range(num_departures)]
target_random_list = None # random_positions # Initialize target_random_list here

# start and cancel buttons
start_b = widgets.Button(description=' Start', layout=Layout(width='50%', height='80px'), disabled=False, button_style='success')
start_rand_b = widgets.Button(description=' Start random', layout=Layout(width='50%', height='80px'), disabled=False, button_style='success')

cancel_b = widgets.Button(description='Cancel target', layout=Layout(width='50%', height='80px'), disabled=True, button_style='danger')

# initialization of the manager target
start_b = start_b
cancel_b = cancel_b
x_input = x_input
y_input = y_input
rand_start_b = start_rand_b

print(random_positions)


# rospy.Timer(rospy.Duration(0.1), update_widgets)
# # rospy.Timer(rospy.Duration(0.1), update_toggle)



# wdget of the closest obstacle and position
box_pos = HBox([label_pos, text_x, text_y], layout=widgets.Layout(justify_content='flex-start', align_items='center', width='100%'))
box_ob = HBox([label_ob, text_ob, label_metres], layout=widgets.Layout(justify_content='flex-start', align_items='center', width='100%'))

# binding buttons to functions
start_b.on_click(on_start_button_clicked)
cancel_b.on_click(on_cancel_button_clicked)
start_rand_b.on_click(on_start_rand_button_clicked)



display(box_input)
display(HBox([start_b, cancel_b]))
display(HBox([box_pos, box_ob]))
display(start_rand_b)
plt.show()

box_layout = Layout(border='1px solid black', width='33%', align_items='center', padding='10px')
display(HBox([
    VBox([widgets.HTML(f'<b> Target cancelled </b>'), cancelled_list_box], layout=box_layout),
    VBox([widgets.HTML(f'<b> Target given </b>'), given_list_box], layout=box_layout),
    VBox([widgets.HTML(f'<b> Target reached </b>'), reached_list_box], layout=box_layout)
], layout=Layout(width='100%')))

# display(widgets.VBox([
#     times_recap  # mostra il riepilogo dei tempi
# ]))

display(feed)
display(sono_qui)
display(teleport_widget)
display(laod_ers_w)
display(reached_w)
display(cancelled_w)




None


Button(button_style='success', description=' Start random', layout=Layout(height='80px', width='50%'), style=B…

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Label(value='feed:')

Label(value='')

Label(value='')

Label(value='')

Label(value='')

Label(value='')

In [12]:
import yaml
# Carica i dati dal file YAML
with open("random_targets_generated.yaml", "r") as f:
    loaded_data_targ = yaml.safe_load(f)  # Use safe_load

with open("random_spawn_positions.yaml", "r") as f:
    loaded_data_spawn = yaml.safe_load(f)  # Use safe_load

random_positions = loaded_data_targ
target_random_list =loaded_data_spawn
random_spawn_position = target_random_list



In [13]:
print(random_positions)
print(target_random_list)

[[-8, -8], [6, 1], [-4, 9], [9, 8], [8, 8], [-7, 2], [5, -8], [-2, -7], [-7, 1], [9, 3], [-5, -9], [-6, -2]]
[[4, 6], [-7, -8], [5, -4], [8, 2], [4, -8], [-1, 7], [-5, 1], [0, -8], [6, 3], [7, 7], [1, -6], [-5, 6], [-2, 2], [-8, 0], [-2, 6], [0, 2], [6, -3], [6, -4], [-5, 7], [8, -1], [5, -1], [7, -3], [-6, -3], [-8, -7]]


In [14]:
print(target_random_list)

[[4, 6], [-7, -8], [5, -4], [8, 2], [4, -8], [-1, 7], [-5, 1], [0, -8], [6, 3], [7, 7], [1, -6], [-5, 6], [-2, 2], [-8, 0], [-2, 6], [0, 2], [6, -3], [6, -4], [-5, 7], [8, -1], [5, -1], [7, -3], [-6, -3], [-8, -7]]
